#### Import

In [1]:
from elasticsearch import Elasticsearch
import json

es = Elasticsearch("http://localhost:9200")

#### Requête 1 : MATCH_ALL = juste récupérer tous les documents (baseline)

In [ ]:
# Requête 1 : juste récupérer tout, sans filtrer (baseline quoi) (MATCH_ALL)
res = es.search(
     index="movies_clean",
     query={"match_all": {}}
)

print(json.dumps(res["hits"]["hits"], indent=2))


[
  {
    "_index": "movies_clean",
    "_id": "235085",
    "_score": 1.0,
    "_ignored": [
      "overview.keyword"
    ],
    "_source": {
      "id": 235085,
      "vote_count": 5,
      "original_language": "en",
      "production_companies": "Epic Productions-Smoking Gun Productions",
      "overview": "Claudia a girl who gets around finally lands her dream man in Mickey. However Mickey's friend Dobbs spoils their nuptial plans when the thugs of a Vietnamese gangster that he has robbed come looking for him. In order to save Dobbs Mickey and Claudia hide him in the boot of their car as they take off to LA. Things don't go as smoothly as planned and Mickey finds out more about his fianc\u00e9e than he ever needs to know.",
      "genres": [
        "Action",
        "Drama"
      ],
      "revenue": 0,
      "tagline": null,
      "log": {
        "file": {
          "path": "/data/movies.csv"
        }
      },
      "runtime": 90.0,
      "credits": "D.B. Sweeney-Bridget Fonda-C

### Chercher des films qui parlent de "vampire" dans leur synopsis.

In [ ]:
# Requête 2 : chercher "vampire" dans le résumé des films (même si c'est pas écrit exactement pareil) (MATCH)
query = {
    "query": {
        "match": {
            "overview": "vampire"
        }
    }
}
res = es.search(index="movies_clean", body=query)
print(f"Total :{res['hits']['total']['value']}")
for hit in res["hits"]["hits"]:
    print(f"- {hit['_source']['title']}")


Total :1159
- Night
- Liar, Liar, Vampire
- Vampire Sisters 3: Journey to Transylvania
- Dracula - Die wahre Geschichte der Vampire
- Dracula's Curse
- Female Vampire
- Vampire Hunter D
- Protege Moi
- Vampire Black: Trail of the Dead
- Hard Times for Dracula


### Chercher uniquement les films dont la langue originale est le japonais.

In [ ]:
# Requête 3 : chercher les films japonais, genre vraiment japonais pas d'approximation / TERM = Val exact
query = {
    "query": {
        "term": {
            "original_language.keyword": "ja"
        }
    }
}
res = es.search(index="movies_clean", body=query)
print(f"Total :{res['hits']['total']['value']}")
for hit in res["hits"]["hits"]:
    print(f"- {hit['_source']['title']}")


Total :10000
- Kiss Him, Not Me
- Mud of Love
- Informe
- Night and Fog in Japan
- Angel Guts: Red Flash
- The Legend of the White Serpent
- Kaitô Ruby
- Gdleen
- Until the Break of Dawn
- Prince of Legend


### Chercher des films de "Comedy" qui ont un budget supérieur à 5 millions.

In [ ]:
# Requête 4 : combiner deux critères --> c'est une comédie ET elle a coûté plus de 5 millions (BOOL)
query = {
    "query": {
        "bool": {
            "must": [
                { "match": { "genres": "Comedy" } }
            ],
            "filter": [
                { "range": { "budget": { "gt": 5000000 } } }
            ]
        }
    }
}
res = es.search(index="movies_clean", body=query)
print(f"Total :{res['hits']['total']['value']}")
for hit in res["hits"]["hits"]:
    print(f"- {hit['_source']['title']}")


Total :2546
- Ninguém Entra, Ninguém Sai
- Two Dads and One Mom
- Lou!
- I-See-You.com
- The Trotsky
- Tutto tutto niente niente
- Mrs. Brown's Boys D'Movie
- Um Suburbano Sortudo
- Dany
- Locked Out


### Chercher des films d' "Action" mais exclure ceux qui durent plus de 2 heures (120 min).

In [ ]:
# Requête 5 : chercher les films d'action MAIS pas trop longs (exclure > 120 min) (BOOL)
query = {
    "query": {
        "bool": {
            "must": [
                { "match": { "genres": "Action" } }
            ],
            "must_not": [
                { "range": { "runtime": { "gt": 120 } } }
            ]
        }
    }
}
res = es.search(index="movies_clean", body=query)
print(f"Total :{res['hits']['total']['value']}")
for hit in res["hits"]["hits"]:
    print(f"- {hit['_source']['title']}")


Total :10000
- Boy Condenado
- Bellator 262: Velasquez vs. Kielholtz
- Four Real Friends
- The Black Tavern
- Executor
- Rumble
- El de los lentes Carrera 2
- White Rush
- 11 Blocks
- UFC 227: Dillashaw vs. Garbrandt 2


### Chercher "Harry Potter" même si on écrit mal "Hary".

In [ ]:
# Requête 6 : trouver Harry Potter même si mal écrit. (MATCH) / FUZZINESS =>
query = {
    "query": {
        "match": {
            "title": {
                "query": "Hary Potter",
                "fuzziness": "AUTO"
            }
        }
    }
}
res = es.search(index="movies_clean", body=query)
print(f"Total :{res['hits']['total']['value']}")
for hit in res["hits"]["hits"]:
    print(f"- {hit['_source']['title']}")


Total :2910
- Pitter-Patter
- Harry Potter: Different Perspective
- Harry Potter: Witchcraft Repackaged
- Twinkle-Twinkle Pitter-Patter
- Daniel Radcliffe: Being Harry Potter
- 50 Greatest Harry Potter Moments
- 'Harry Potter': Behind the Magic
- The Harry Potter Saga Analyzed
- Pitter Patter Goes My Heart
- Hari Hara Veera Mallu


### Les meilleurs films français récents et bien notés
On veut trouver les bons drames français de ces dernières années, des films avec une vraie qualité.

In [ ]:
# Requête 7 : on veut les dramas français, bien notés, et récents (après 2020)
# on cumule tous les critères ensemble, faut que ce soit du drama ET bien noté ET français ET récent
query = {
    "query": {
        "bool": {
            "must": [
                { "match": { "genres": "Drama" } },
                { "range": { "vote_average": { "gte": 7 } } }
            ],
            "filter": [
                { "term": { "original_language.keyword": "fr" } },
                { "range": { "release_date": { "gte": "2020-01-01" } } }
            ]
        }
    }
}
res = es.search(index="movies_clean", body=query)
print(f"Total : {res['hits']['total']['value']}")
for hit in res["hits"]["hits"][:5]:
    print(f"- {hit['_source']['title']} ({hit['_source']['vote_average']}/10)")


### Les blockbusters d'action qui cartonnent vraiment
Chercher les films d'action qui ont soit gagné beaucoup d'argent, soit qui sont hyper populaires (ou les deux).

In [ ]:
# Requête 8 : les films d'action qu'ont vraiment percé, soit ils ont fait beaucoup d'argent SOIT beaucoup de gens en parlent (BOOL)
# l'un ou l'autre (ou les deux c'est encore mieux) = c'est un sacré blockbusters 
query = {
    "query": {
        "bool": {
            "must": [
                { "match": { "genres": "Action" } }
            ],
            "should": [
                { "range": { "revenue": { "gt": 100000000 } } },
                { "range": { "popularity": { "gt": 50 } } }
            ],
            "minimum_should_match": 1
        }
    }
}
res = es.search(index="movies_clean", body=query)
print(f"Total : {res['hits']['total']['value']}")
for hit in res["hits"]["hits"][:5]:
    print(f"- {hit['_source']['title']} (Revenue: ${hit['_source']['revenue']:,.0f})")


### Les documentaires faits sans gros budget
Trouver les documentaires indépendants, ceux qui ont été tournés sans financement énorme, juste avec de la volonté.

In [ ]:
# Requête 9 : les documentaires vraiment indé (auncun budge), c'est des films faits par passion. (BOOL)
query = {
    "query": {
        "bool": {
            "must": [
                { "match": { "genres": "Documentary" } }
            ],
            "must_not": [
                { "range": { "budget": { "gt": 0 } } }
            ]
        }
    }
}
res = es.search(index="movies_clean", body=query)
print(f"Total : {res['hits']['total']['value']}")
for hit in res["hits"]["hits"][:10]:
    print(f"- {hit['_source']['title']}")


### Les films d'horreur qui ont bien plu aux gens
On cherche les films d'horreur avec une bonne note (donc au moins 6/10). Les films qui ont vraiment fait peur mais dans le bon sens.

In [ ]:
# Requête 10 : les films d'horreur qui font vraiment peur et qui sont bien (note >= 6/10), (pas les mauvais nanar)
query = {
    "query": {
        "bool": {
            "must": [
                { "match": { "genres": "Horror" } },
                { "range": { "vote_average": { "gte": 6 } } }
            ]
        }
    }
}
res = es.search(index="movies_clean", body=query)
print(f"Total : {res['hits']['total']['value']}")
for hit in res["hits"]["hits"][:10]:
    print(f"- {hit['_source']['title']} ({hit['_source']['vote_average']}/10)")


### Les films de science-fiction à fort Revenu
Chercher les sci-fi films qui ont vraiment marché au box-office, ceux qui ont généré au moins 50 millions.

In [ ]:
# Requête 11 : les films de sci-fi qui ont vraiment gagné beaucoup d'argent (> 50 millions c'est un vrai succès). (BOOL)
query = {
    "query": {
        "bool": {
            "must": [
                { "match": { "genres": "Science Fiction" } }
            ],
            "filter": [
                { "range": { "revenue": { "gte": 50000000 } } }
            ]
        }
    }
}
res = es.search(index="movies_clean", body=query)
print(f"Total : {res['hits']['total']['value']}")
for hit in res["hits"]["hits"][:10]:
    print(f"- {hit['_source']['title']} (${hit['_source']['revenue']:,.0f})")


### Les films de romance qui ont du succès
On veut les films de romance avec une bonne notation ET un bon revenue, genre les grands succès sentimentaux.

In [ ]:
# Requête 12 : les vrais succès de romance = bien noté ET ça a rapporté de l'argent (> 20 millions) (BOOL)
# Les films que tout le monde a aimé ET qui a marché au box-office 
query = {
    "query": {
        "bool": {
            "must": [
                { "match": { "genres": "Romance" } },
                { "range": { "vote_average": { "gte": 7 } } }
            ],
            "filter": [
                { "range": { "revenue": { "gt": 20000000 } } }
            ]
        }
    }
}
res = es.search(index="movies_clean", body=query)
print(f"Total : {res['hits']['total']['value']}")
for hit in res["hits"]["hits"][:10]:
    print(f"- {hit['_source']['title']} ({hit['_source']['vote_average']}/10, ${hit['_source']['revenue']:,.0f})")
